# Setup: Download Model

Downloads `opendatalab/MinerU2.5-2509-1.2B` from HuggingFace Hub to a Unity Catalog Volume. This is a one-time setup step shared by all deployment options.

**Idempotent** — skips download if `config.json` already exists at the target path.

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | Serverless |
| Libraries | None — installed via `%pip` below |

### Prerequisites
- Unity Catalog catalog and schema must exist (the Volume is created automatically)
- HuggingFace network access from the cluster (set `HF_TOKEN` env var if the model is gated)

### Install Dependencies

`huggingface_hub` provides `snapshot_download` for efficient model download.
`hf_transfer` enables the Rust-based high-speed transfer backend.

In [ ]:
%pip install huggingface_hub hf_transfer

In [ ]:
import yaml, os

# Resolve the project root directory.
# When running as a Databricks notebook, __file__ is not defined,
# so we fall back to the notebook context API to get the workspace path.
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

# Load centralised configuration — all notebooks read from the same config.yaml
# to avoid hardcoded catalog/schema/volume values.
cfg        = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG    = cfg["catalog"]
SCHEMA     = cfg["schema"]
VOLUME     = cfg["volume"]
MODEL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{cfg['model_subpath']}"
HF_ID      = cfg["hf_model_id"]

# Create the UC Volume if it doesn't exist yet.
# os.makedirs won't work here — UC Volumes must be created via SQL.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

print(f"MODEL_PATH : {MODEL_PATH}")
print(f"HF_ID      : {HF_ID}")

### Download Model from HuggingFace

Uses `snapshot_download` to pull the full model repository. The `HF_HUB_ENABLE_HF_TRANSFER=1` flag activates the Rust-based transfer backend for ~3-5x faster downloads.

The download is **idempotent**: if `config.json` already exists at `MODEL_PATH`, the cell is skipped entirely.

In [ ]:
import os
from huggingface_hub import snapshot_download

# Check if model is already downloaded by looking for config.json
config_file = os.path.join(MODEL_PATH, "config.json")

if os.path.exists(config_file):
    print(f"Skipping download — config.json already exists at {config_file}")
else:
    print(f"Downloading {HF_ID} to {MODEL_PATH} ...")
    # Enable Rust-based transfer for faster downloads (~3 GB model)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    snapshot_download(repo_id=HF_ID, local_dir=MODEL_PATH)
    print("Download complete.")

# Verify: list all downloaded files
print("\nFiles in MODEL_PATH:")
for f in sorted(os.listdir(MODEL_PATH)):
    print(f"  {f}")